# 02 · Intraday patterns

The U-shape in volume and volatility is famous enough to be assumed. Here it is
measured, so that "a big move in the final 30 minutes" can be read against what
the final 30 minutes normally look like.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from closingbell import config as C, calendar_utils as cal

def table(name):
    return pd.read_csv(C.TABLES / f"{name}.csv")

sessions = pd.read_parquet(C.PROCESSED / "sessions.parquet")
print(f"{len(sessions):,} ticker-sessions, {sessions.session.min()} to {sessions.session.max()}")

15,756 ticker-sessions, 2021-01-04 to 2026-03-31


## The normalised profile

Each ticker is scaled by its own all-day average, so an ETF and a single name are comparable in shape.

In [2]:
prof = table('intraday_profile')
cross = table('intraday_profile_cross_section').sort_values("slot")
cross[["slot", "median_volume_norm", "mean_abs_return_norm", "mean_hl_range_norm"]].round(3).head(20)

,slot,median_volume_norm,mean_abs_return_norm,mean_hl_range_norm
0,09:30,3.836,NaN,3.170
1,09:35,2.164,2.253,2.223
2,09:40,1.907,2.023,1.967
3,09:45,1.823,1.926,1.877
4,09:50,1.628,1.726,1.696
5,09:55,1.484,1.520,1.506
6,10:00,1.640,1.734,1.716
7,10:05,1.424,1.516,1.492
8,10:10,1.357,1.507,1.457
9,10:15,1.297,1.379,1.360


In [3]:
import matplotlib.pyplot as plt
from closingbell import plots
fig, ax = plt.subplots(figsize=(9, 3.6))
c = cross.sort_values("slot")
ax.plot(range(len(c)), c.median_volume_norm, lw=2, color=plots.ACCENT, label="volume")
ax.plot(range(len(c)), c.mean_abs_return_norm, lw=2, color=plots.DOWN, label="|return|")
ax.axhline(1.0, color="k", lw=0.8, ls="--"); ax.legend(); ax.set_ylabel("x daily average")
plots._slot_ticks(ax, sorted(c.slot)); ax.set_title("volume vs volatility through the day", loc="left")
plt.show()

The two curves separate sharply in the afternoon: volume climbs far more steeply into the close than price movement does.

## Quantifying the U

In [4]:
u = table('u_shape_diagnostics')
cols = [c for c in u.columns if c != "ticker"]
u[cols].mean().round(3).to_frame("cross-ticker mean")

,cross-ticker mean
median_volume_norm_open,3.836
median_volume_norm_midday,0.686
median_volume_norm_close,4.557
median_volume_norm_close30,1.954
median_volume_norm_close_over_midday,6.721
median_volume_norm_open_over_midday,5.601
mean_abs_return_norm_open,2.253
mean_abs_return_norm_midday,0.842
mean_abs_return_norm_close,1.174
mean_abs_return_norm_close30,0.992


Read the ratios: volume at the close runs about **6.7x** the midday trough,
while mean absolute return runs only about **1.4x**. Order flow concentrates at
the close much more than price movement does.

## How much of the day trades in the final 30 minutes?

In [5]:
cs = table('closing_volume_share')
print(cs.round(4).to_string(index=False))
print()
print("median close30 share (incl. auction): %.1f%%" % (100 * cs.close30_share.median()))
print("median auction share:                 %.1f%%" % (100 * cs.auction_share.median()))

ticker  close30_share_ex_auction  close30_share  auction_share
  AAPL                    0.1321         0.2496         0.1306
  AMZN                    0.1279         0.2220         0.1053
 GOOGL                    0.1530         0.2718         0.1341
   IWM                    0.1519         0.1760         0.0235
   JPM                    0.1681         0.3103         0.1659
  META                    0.1212         0.2007         0.0888
  MSFT                    0.1485         0.2989         0.1719
  NVDA                    0.0990         0.1531         0.0551
   QQQ                    0.1368         0.1530         0.0150
   SPY                    0.1920         0.2179         0.0286
  TSLA                    0.0823         0.1108         0.0274
   XOM                    0.1909         0.2051         0.0000

median close30 share (incl. auction): 21.2%
median auction share:                 7.2%


## The distribution of closing pressure

In [6]:
z = sessions.closing_pressure_z.dropna()
print(f"n = {len(z):,}")
print(z.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(3))
print()
print("excess kurtosis: %.2f (normal = 0)" % (z.kurtosis()))
print("share beyond +/-3 sigma: %.2f%% (normal = 0.27%%)" % (100 * (z.abs() > 3).mean()))

n = 15,088
count    15088.000
mean         0.004
std          1.063
min         -8.883
1%          -3.013
5%          -1.657
25%         -0.553
50%          0.025
75%          0.591
95%          1.641
99%          2.739
max          6.431
Name: closing_pressure_z, dtype: float64

excess kurtosis: 4.42 (normal = 0)
share beyond +/-3 sigma: 1.74% (normal = 0.27%)


Fat tails are why events are defined by trailing percentile rather than by a fixed z threshold.

See `results/figures/fig01`, `fig02` and `fig03`.